# 01 · Carga de noticias a DataFrame y exportación a parquet

Toma la salida del scraper (`data/raw/noticias.jsonl`), le asigna tipos, agrega columnas derivadas útiles para el análisis y guarda el resultado en `data/processed/noticias.parquet`.

Se parte del JSONL y no del CSV porque el cuerpo de las notas tiene saltos de línea y comillas; JSONL los conserva sin ambigüedad.

In [1]:
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw" / "noticias.jsonl"
SALIDA = RAIZ / "data" / "processed" / "noticias.parquet"

pd.set_option("display.max_colwidth", 80)
RAW

PosixPath('/Users/macair/Documents/Maestria/PLN/Actividad 1/data/raw/noticias.jsonl')

## 1. Carga

In [2]:
df = pd.read_json(RAW, lines=True, dtype=False)
print(f"{len(df):,} notas, {df.shape[1]} columnas")
df.head(3)

23,856 notas, 11 columnas


,id,url,titulo,fecha,fecha_texto,autor,autor_iniciales,sumario,imagen,cuerpo,n_palabras
0,14264,https://www.unison.mx/nota/?idnoti=14264,Desarrollan alumnos técnica aprobada por OMS para un mejor diagnóstico del l...,2013-01-01,01 de enero de 2013,NaN,NaN,NaN,https://www.unison.mx/noticias/img-gde/14264.jpg,Estudiantes del Departamento de Ciencias Químico Biológicas de la Universida...,361
1,14265,https://www.unison.mx/nota/?idnoti=14265,Registro en línea para examen de colocación de inglés será el 4 de enero de ...,2013-01-02,02 de enero de 2013,NaN,NaN,NaN,https://www.unison.mx/noticias/img-gde/14265.jpg,El Departamento de Lenguas Extranjeras de la Universidad de Sonora emitió la...,322
2,14266,https://www.unison.mx/nota/?idnoti=14266,Interesa en Brasil Sistema de Servicios Sustentables que desarrollan académi...,2013-01-03,03 de enero de 2013,NaN,NaN,NaN,https://www.unison.mx/noticias/img-gde/14266.jpg,Empresarios del ramo automotriz dedicados a la reparación de carrocería en B...,412


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23856 entries, 0 to 23855
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               23856 non-null  int64
 1   url              23856 non-null  str  
 2   titulo           23856 non-null  str  
 3   fecha            23856 non-null  str  
 4   fecha_texto      23856 non-null  str  
 5   autor            11157 non-null  str  
 6   autor_iniciales  8216 non-null   str  
 7   sumario          100 non-null    str  
 8   imagen           23856 non-null  str  
 9   cuerpo           23856 non-null  str  
 10  n_palabras       23856 non-null  int64
dtypes: int64(2), str(9)
memory usage: 73.5 MB


## 2. Tipos

- `fecha` a `datetime64`.
- `id` y `n_palabras` a enteros de 32 bits.
- Los textos quedan como `str` (el tipo de texto de pandas 3); `autor` y `autor_iniciales` como `category`, porque se repiten mucho.

In [4]:
df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d")
df["id"] = df["id"].astype("int32")
df["n_palabras"] = df["n_palabras"].astype("int32")

for col in ["url", "titulo", "fecha_texto", "sumario", "imagen", "cuerpo"]:
    df[col] = df[col].astype("str")
for col in ["autor", "autor_iniciales"]:
    df[col] = df[col].astype("category")

df.dtypes

id                          int32
url                           str
titulo                        str
fecha              datetime64[us]
fecha_texto                   str
autor                    category
autor_iniciales          category
sumario                       str
imagen                        str
cuerpo                        str
n_palabras                  int32
dtype: object

## 3. Columnas derivadas

- **Tiempo**: `anio`, `mes`, `trimestre`, `semestre`, `dia_semana`. Sirven para estacionalidad y evolución temporal.
- **Firma**: `firma` toma el nombre del autor y, si no hay, sus iniciales (notas antiguas). `tiene_firma` lo resume.
- **Tamaño**: `n_parrafos` y `n_caracteres`.
- **`texto`**: título + sumario + cuerpo, la entrada para el procesamiento de NLP.

In [5]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

df["anio"] = df["fecha"].dt.year.astype("int16")
df["mes"] = df["fecha"].dt.month.astype("int8")
df["trimestre"] = df["fecha"].dt.quarter.astype("int8")
df["semestre"] = ((df["mes"] > 6).astype("int8") + 1)
df["dia_semana"] = pd.Categorical(df["fecha"].dt.dayofweek.map(dict(enumerate(DIAS))),
                                  categories=DIAS, ordered=True)

df["firma"] = (df["autor"].astype("str")
               .fillna(df["autor_iniciales"].astype("str"))
               .astype("category"))
df["tiene_firma"] = df["firma"].notna()

df["n_parrafos"] = df["cuerpo"].str.count("\n").add(1).astype("int16")
df["n_caracteres"] = df["cuerpo"].str.len().astype("int32")

df["texto"] = (df["titulo"] + "\n" + df["sumario"].fillna("") + "\n" + df["cuerpo"]).str.replace(r"\n{2,}", "\n", regex=True)

df[["fecha", "anio", "mes", "dia_semana", "firma", "n_parrafos", "n_caracteres"]].sample(5, random_state=0)

,fecha,anio,mes,dia_semana,firma,n_parrafos,n_caracteres
5306,2015-08-03,2015,8,lunes,EQ,9,2623
15537,2021-01-14,2021,1,jueves,Lin Mendivil Alvarado,11,4461
10476,2017-11-09,2017,11,jueves,NaN,11,3176
12350,2018-10-11,2018,10,jueves,Lin Mendivil Alvarado,6,2102
9755,2017-07-07,2017,7,viernes,NaN,8,2564


## 4. Calidad

Revisión de nulos, duplicados y notas demasiado cortas.

- Las notas publicadas dos veces (mismo título y cuerpo) se eliminan, conservando la de ID menor.
- Las notas con menos de 20 palabras no se eliminan, solo se marcan con `es_corta`, para decidir en el análisis.

In [6]:
df.isna().sum()[lambda s: s > 0]

autor              12699
autor_iniciales    15640
sumario            23756
firma               4484
dtype: int64

In [7]:
dup = df.duplicated(["titulo", "cuerpo"], keep="first")
print(f"Duplicadas (título + cuerpo): {dup.sum()}")
df[df.duplicated(["titulo", "cuerpo"], keep=False)].sort_values(["titulo", "id"])[["id", "fecha", "titulo"]].head(6)

Duplicadas (título + cuerpo): 26


,id,fecha,titulo
2706,17023,2014-06-16,Alertan sobre deficiencia de Vitamina A en la población infantil menor de ci...
2945,17273,2014-07-21,Alertan sobre deficiencia de Vitamina A en la población infantil menor de ci...
5243,19643,2015-07-07,"Alumno del Campus Caborca estudiará en universidad de Buenos Aires, Argentina"
5276,19678,2015-07-20,"Alumno del Campus Caborca estudiará en universidad de Buenos Aires, Argentina"
2238,16543,2014-02-28,Analizan la volatilidad económica que se presenta en las economías emergentes
2242,16547,2014-02-28,Analizan la volatilidad económica que se presenta en las economías emergentes


In [8]:
df = df.sort_values("id")
df = df[~df.duplicated(["titulo", "cuerpo"], keep="first")]

df["es_corta"] = df["n_palabras"] < 20
print(f"Notas cortas (<20 palabras): {df['es_corta'].sum()}")

df = df.sort_values(["fecha", "id"]).reset_index(drop=True)
print(f"{len(df):,} notas tras la limpieza")

Notas cortas (<20 palabras): 19
23,830 notas tras la limpieza


## 5. Resumen por año

In [9]:
(df.groupby("anio")
   .agg(notas=("id", "size"),
        con_firma=("tiene_firma", "mean"),
        palabras_mediana=("n_palabras", "median"),
        autores=("firma", "nunique"))
   .round(2))

,notas,con_firma,palabras_mediana,autores
anio,,,,
2013,1899,0.10,314.0,14
2014,2136,0.67,356.0,25
2015,2375,0.83,370.0,26
2016,2336,0.87,381.0,26
2017,2051,0.79,374.0,28
2018,2053,0.85,382.0,48
2019,1697,0.93,435.0,27
2020,946,0.94,516.0,24
2021,1355,0.94,587.0,32


## 6. Exportación a parquet

In [10]:
COLUMNAS = ["id", "fecha", "anio", "mes", "trimestre", "semestre", "dia_semana",
            "titulo", "sumario", "cuerpo", "texto",
            "autor", "autor_iniciales", "firma", "tiene_firma",
            "n_palabras", "n_parrafos", "n_caracteres", "es_corta",
            "fecha_texto", "url", "imagen"]
df = df[COLUMNAS]

SALIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(SALIDA, engine="pyarrow", compression="zstd", index=False)
print(f"{SALIDA.relative_to(RAIZ)}: {SALIDA.stat().st_size / 1e6:.1f} MB "
      f"(JSONL original: {RAW.stat().st_size / 1e6:.1f} MB)")

data/processed/noticias.parquet: 51.9 MB (JSONL original: 79.4 MB)


Verificación: se vuelve a leer el parquet y se compara con el DataFrame en memoria.

In [11]:
df_leido = pd.read_parquet(SALIDA)
pd.testing.assert_frame_equal(df, df_leido)
print("El parquet coincide con el DataFrame:", df_leido.shape)
df_leido.dtypes

El parquet coincide con el DataFrame: (23830, 22)


id                          int32
fecha              datetime64[us]
anio                        int16
mes                          int8
trimestre                    int8
semestre                     int8
dia_semana               category
titulo                        str
sumario                       str
cuerpo                        str
texto                         str
autor                    category
autor_iniciales          category
firma                    category
tiene_firma                  bool
n_palabras                  int32
n_parrafos                  int16
n_caracteres                int32
es_corta                     bool
fecha_texto                   str
url                           str
imagen                        str
dtype: object

In [12]:
df_leido.head()

,id,fecha,anio,mes,trimestre,semestre,dia_semana,titulo,sumario,cuerpo,...,autor_iniciales,firma,tiene_firma,n_palabras,n_parrafos,n_caracteres,es_corta,fecha_texto,url,imagen
0,14264,2013-01-01,2013,1,1,1,martes,Desarrollan alumnos técnica aprobada por OMS para un mejor diagnóstico del l...,NaN,Estudiantes del Departamento de Ciencias Químico Biológicas de la Universida...,...,NaN,NaN,False,361,7,2298,False,01 de enero de 2013,https://www.unison.mx/nota/?idnoti=14264,https://www.unison.mx/noticias/img-gde/14264.jpg
1,14265,2013-01-02,2013,1,1,1,miércoles,Registro en línea para examen de colocación de inglés será el 4 de enero de ...,NaN,El Departamento de Lenguas Extranjeras de la Universidad de Sonora emitió la...,...,NaN,NaN,False,322,7,1919,False,02 de enero de 2013,https://www.unison.mx/nota/?idnoti=14265,https://www.unison.mx/noticias/img-gde/14265.jpg
2,14266,2013-01-03,2013,1,1,1,jueves,Interesa en Brasil Sistema de Servicios Sustentables que desarrollan académi...,NaN,Empresarios del ramo automotriz dedicados a la reparación de carrocería en B...,...,NaN,NaN,False,412,9,2579,False,03 de enero de 2013,https://www.unison.mx/nota/?idnoti=14266,https://www.unison.mx/noticias/img-gde/14266.jpg
3,14267,2013-01-03,2013,1,1,1,jueves,Aplica académico el enfoque psicológico a la sustentabilidad,NaN,El enfoque psicológico de la sustentabilidad que permita enfrentar y abordar...,...,NaN,NaN,False,241,6,1588,False,03 de enero de 2013,https://www.unison.mx/nota/?idnoti=14267,https://www.unison.mx/noticias/img-gde/14267.jpg
4,14268,2013-01-04,2013,1,1,1,viernes,Serán del 7 al 11 de enero las reinscripciones generales para semestre 2013-1,NaN,El proceso de reinscripciones para el semestre 2013-1 se realizará del 7 al ...,...,NaN,NaN,False,358,10,2094,False,04 de enero de 2013,https://www.unison.mx/nota/?idnoti=14268,https://www.unison.mx/noticias/img-gde/14268.jpg


## Uso en los siguientes notebooks

```python
import pandas as pd
df = pd.read_parquet("../data/processed/noticias.parquet")
```